# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

Notebook **khung** cho Colab hoặc Kaggle. Các ô có chữ `TODO` là phần **bạn tự viết**; ô cài đặt, tải dữ liệu
và chạy `eval.py` đã viết sẵn. Đọc `README.md` (quy tắc chia dữ liệu, cách đánh giá), `GUIDE.md` và `RUBRIC.md` trước.

**Quy tắc quan trọng nhất:** chọn mọi thứ trên **val**; **test chỉ chạy một lần mỗi seed ở Bước 4**.

Bản đồ: Bước 0 (EDA, kiểm tra) → Bước 1 (backbone) → Bước 2 (công thức huấn luyện) → Bước 3 (suy luận)
→ Bước 4 (chung kết + `eval.py`) → Bước 5 (xlsx, biểu đồ, báo cáo).

## 0. Cài đặt môi trường

In [ ]:
# Cài thư viện (Colab/Kaggle). Kaggle: bật Internet trong Settings.
!pip -q install timm openpyxl

import sys, os, platform
import numpy as np, pandas as pd, torch, timm

# Colab/Kaggle: repo đã `git clone` vào thư mục hiện tại. Máy cá nhân: kernel chạy từ submissions/<...>/code/.
REPO_URL = "https://github.com/tam253211-a11y/K4-Track4-Day2-DangHuuTam-2A202602940-Deeplearning-Advance.git"
REPO_DIR = "K4-Track4-Day2-DangHuuTam-2A202602940-Deeplearning-Advance"
if os.path.exists("eval.py"):                # chạy lại ô này trong cùng phiên: đã đứng ở gốc repo
    REPO_DIR = "."
elif not os.path.exists(f"{REPO_DIR}/eval.py") and os.path.exists("../../../eval.py"):
    REPO_DIR = "../../.."
if not os.path.exists(f"{REPO_DIR}/eval.py"):    # Colab/Kaggle mở notebook lẻ: clone repo về
    !git clone -q {REPO_URL} {REPO_DIR}
assert os.path.exists(f"{REPO_DIR}/eval.py"), "Không tìm thấy gốc repo, sửa REPO_DIR"
os.chdir(REPO_DIR)                           # từ đây mọi đường dẫn (data/, predictions/...) tính từ gốc repo
REPO_DIR = os.getcwd()
CODE_DIR = f"{REPO_DIR}/submissions/2A202602940_dang_huu_tam/code"
sys.path.insert(0, REPO_DIR)                 # để import eval.py
sys.path.insert(0, CODE_DIR)                 # code/ đã chép từ starter/

print("python", platform.python_version(), "| torch", torch.__version__, "| timm", timm.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")

### Tải dữ liệu (đã viết sẵn)
Ảnh từ Zenodo (~490 MB, kiểm tra MD5), nhãn và các file chia fold từ GitHub của tác giả.
**Sau khi giải nén, hãy `ls` để xem ảnh nằm ở thư mục nào** rồi đặt `IMAGES_DIR` cho đúng.

In [ ]:
import hashlib

os.makedirs("data/labels", exist_ok=True)
if not os.path.exists("data/images.zip"):
    !wget -q -O data/images.zip "https://zenodo.org/records/7939060/files/images.zip?download=1"

h = hashlib.md5()
with open("data/images.zip", "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", f"MD5 sai: {h.hexdigest()}"

!unzip -q -n data/images.zip -d data/images/   # zip không có thư mục con: .jpg nằm thẳng trong data/images/
!ls data/images | head -3; ls data/images | wc -l

BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    !wget -q -O data/labels/{name}.csv {BASE}/{name}.csv
!ls -la data/labels

In [ ]:
IMAGES_DIR = "data/images"   # TODO: kiểm tra đúng thư mục chứa các file .jpg sau khi giải nén
LABELS_DIR = "data/labels"

### Nơi lưu bền: Google Drive
`/content` bị xoá khi Colab ngắt phiên. Ô dưới nối `runs/` (checkpoint, log, logit), `predictions/`, `curves/`, `eval_out/`
sang Drive, nên mọi thứ code ghi vào các thư mục này nằm thẳng trên Drive. Chạy lại ô này sau mỗi lần phiên khởi động lại.

In [ ]:
OUTPUT_DIRS = ["runs", "predictions", "curves", "eval_out"]
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST_DIR = "/content/drive/MyDrive/K4-Day2-DeepWeeds"
elif os.path.isdir("/kaggle/working"):
    PERSIST_DIR = "/kaggle/working"           # Kaggle: lưu thành output khi Save & Run All
else:
    PERSIST_DIR = None                        # máy cá nhân: ổ đĩa đã bền sẵn
if PERSIST_DIR:
    for d in OUTPUT_DIRS:
        target = f"{PERSIST_DIR}/{d}"
        os.makedirs(target, exist_ok=True)
        if not os.path.islink(d):
            assert not os.path.exists(d) or not os.listdir(d), f"{d}/ đã có dữ liệu cục bộ, chép sang {target} trước"
            if os.path.exists(d):
                os.rmdir(d)
            os.symlink(target, d)
for d in OUTPUT_DIRS:
    os.makedirs(d, exist_ok=True)
    print(f"{d:12s} -> {os.path.realpath(d)}")


## Bước 0 — EDA và kiểm tra pipeline
Mục tiêu: hiểu dữ liệu, chạy các kiểm tra chia dữ liệu (README mục 2.1), kiểm tra pipeline trước khi chạy thật.

In [ ]:
import dataset
train_df, val_df, test_df = dataset.load_split(LABELS_DIR, fold=0)
split_stats = dataset.check_split(train_df, val_df, test_df, IMAGES_DIR)


In [ ]:
# EDA: phân bố lớp, đối chiếu Table 1 của bài báo, kích thước/kênh ảnh, ảnh mẫu (hình lưu ở eval_out/ trên Drive)
import matplotlib.pyplot as plt
from PIL import Image

pc = split_stats["per_class"]
ax = pc[["train", "val", "test"]].plot.bar(stacked=True, figsize=(10, 4), title="DeepWeeds fold 0: số ảnh mỗi lớp")
ax.set_ylabel("số ảnh"); plt.xticks(rotation=30, ha="right"); plt.tight_layout()
plt.savefig("eval_out/eda_class_distribution.png", dpi=120); plt.show()

tot = pc["total"]
print(f"Lớp nhiều nhất / ít nhất = {tot.max()} / {tot.min()} = {tot.max() / tot.min():.2f} lần;"
      f" Negatives chiếm {tot.iloc[-1] / tot.sum():.1%}")
paper = [1125, 1064, 1031, 1022, 1062, 1009, 1074, 1016, 9106]  # Table 1, Olsen et al. 2019
cmp = pd.DataFrame({"đếm được": tot.values, "bài báo": paper}, index=tot.index)
cmp["chênh"] = cmp["đếm được"] - cmp["bài báo"]
display(cmp)

sample = train_df.sample(500, random_state=0).Filename
info = [(im.size, im.mode) for im in (Image.open(f"{IMAGES_DIR}/{f}") for f in sample)]
print("Kích thước, kênh (500 ảnh train ngẫu nhiên):", pd.Series(info).value_counts().to_dict())
px = np.stack([np.asarray(Image.open(f"{IMAGES_DIR}/{f}").convert("RGB"), dtype=np.float32) / 255 for f in sample[:200]])
print("mean RGB", px.mean((0, 1, 2)).round(3), "std RGB", px.std((0, 1, 2)).round(3),
      "| ImageNet mean", dataset.IMAGENET_MEAN, "std", dataset.IMAGENET_STD)

fig, axes = plt.subplots(9, 4, figsize=(9, 21))
for c in range(9):
    for j, f in enumerate(train_df[train_df.Label == c].sample(4, random_state=0).Filename):
        axes[c, j].imshow(Image.open(f"{IMAGES_DIR}/{f}")); axes[c, j].axis("off")
    axes[c, 0].set_title(f"{c} {dataset.CLASS_NAMES[c]}", loc="left", fontsize=11)
plt.tight_layout(); plt.savefig("eval_out/eda_samples.png", dpi=80); plt.show()


In [ ]:
# Kiểm tra model.py: tag trọng số, #params, GMAC, nhóm tham số, đóng băng, loss ban đầu ~ ln 9
import importlib, math, torch, model as M
importlib.reload(M)
rows = []
for key in M.SUGGESTED_BACKBONES:
    m = M.build_model(key, init="finetune")
    groups = {g["name"]: sum(p.numel() for p in g["params"]) for g in M.param_groups(m, 1e-4, 1e-3, 0.05)}
    rows.append({"backbone": key, "tag": M.weight_tag(m), "params_M": round(M.count_params(m), 2),
                 "GMAC": round(M.count_gmacs(m, 224), 2), **{f"n_{k}": v for k, v in groups.items()}})
    del m
display(pd.DataFrame(rows))

# đóng băng: chỉ head còn requires_grad, BN của backbone ở eval sau set_train_mode
m = M.build_model("resnet50", init="frozen")
M.set_train_mode(m)
bn_train = [n for n, x in m.named_modules() if isinstance(x, torch.nn.BatchNorm2d) and x.training]
trainable = [n for n, p in m.named_parameters() if p.requires_grad]
print("frozen: trainable =", trainable, "| BN còn train mode:", len(bn_train))
assert trainable == ["fc.weight", "fc.bias"] and not bn_train

# loss CE ban đầu với head mới phải gần -ln(1/9) = 2.197
m = M.build_model("resnet50").eval()
x = torch.randn(32, 3, 224, 224); y = torch.randint(0, 9, (32,))
with torch.no_grad():
    loss0 = torch.nn.functional.cross_entropy(m(x), y).item()
print(f"loss ban đầu = {loss0:.3f} (kỳ vọng ~ {math.log(9):.3f})")


In [ ]:
# Kiểm tra losses.py (GUIDE mục 3.2): focal gamma=0 == CE, label smoothing eps=0 == CE, CutMix lam theo diện tích thật
import losses as L
importlib.reload(L)
torch.manual_seed(0)
logits, y = torch.randn(64, 9), torch.randint(0, 9, (64,))
ce = torch.nn.functional.cross_entropy(logits, y)
assert abs(L.FocalLoss(gamma=0)(logits, y) - ce) < 1e-6, "focal gamma=0 phải bằng CE"
assert abs(L.LabelSmoothingCE(0.0)(logits, y) - ce) < 1e-6, "LS eps=0 phải bằng CE"
assert abs(L.LabelSmoothingCE(0.1)(logits, y) - torch.nn.functional.cross_entropy(logits, y, label_smoothing=0.1)) < 1e-6
print("focal(gamma=2) =", L.FocalLoss(2.0)(logits, y).item(), "< CE =", ce.item())
w = L.class_weights(np.bincount(train_df.Label, minlength=9)); print("class weights (1/n):", w.numpy().round(3))
# Mixup/CutMix: ảnh i có toàn bộ pixel = i, nhãn = i -> đọc được trộn với ảnh nào, bao nhiêu diện tích
x, y = torch.arange(8.).view(8, 1, 1, 1).expand(8, 3, 224, 224).clone(), torch.arange(8)
for trial in range(50):                           # nhiều lần để gặp cả hộp bị cắt ở biên
    np.random.seed(trial); torch.manual_seed(trial)
    xm, (ya, yb, lam) = L.mix_batch(x, y, 1.0, "cutmix")
    assert torch.equal(ya, y) and sorted(yb.tolist()) == list(range(8)), "y_b phải là hoán vị của y"
    for i in range(8):
        kept = (xm[i] == i).float().mean().item()   # phần diện tích còn là ảnh gốc
        if yb[i] != i:
            assert abs(kept - lam) < 1e-5, f"CutMix: lam={lam:.4f} nhưng diện tích giữ lại={kept:.4f}"
            assert torch.all((xm[i] == i) | (xm[i] == yb[i])), "vùng dán phải lấy từ đúng ảnh y_b"
xm, (ya, yb, lam) = L.mix_batch(x, y, 0.4, "mixup")
assert torch.allclose(xm, lam * x + (1 - lam) * x[yb]), "Mixup: x_mix = lam*x + (1-lam)*x[perm]"
mixed = L.mixed_loss(torch.nn.functional.cross_entropy, logits[:8], (ya, yb, lam))
ref = lam * torch.nn.functional.cross_entropy(logits[:8], ya) + (1 - lam) * torch.nn.functional.cross_entropy(logits[:8], yb)
assert abs(mixed - ref) < 1e-6, "mixed_loss = lam*CE(y_a) + (1-lam)*CE(y_b)"
print(f"CutMix: 50 lần, lam luôn = diện tích ảnh gốc còn lại (sau khi hộp bị cắt ở biên); y_b = y[perm]. Mixup và mixed_loss đúng.")
print("OK: các kiểm tra loss đều qua")


In [ ]:
# Kiểm tra pipeline (slide trang 59, GUIDE mục 1.3) trước khi chạy thí nghiệm thật
import importlib, math
import dataset as D, model as M, losses as L, train as TR
for mod in (D, M, L, TR):
    importlib.reload(mod)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tf_eval = D.build_transforms(False)

# 1) seed: cùng seed -> cùng thứ tự batch, cùng augmentation, cùng khởi tạo head; seed khác -> khác
def first_batch(seed):
    TR.set_seed(seed)
    head = M.build_model("resnet50").get_classifier().weight.detach().clone()
    dl = D.make_loader(train_df, IMAGES_DIR, D.build_transforms(True), 8, True, num_workers=2, seed=seed)
    x, _, f = next(iter(dl))
    return x, list(f), head
(xa, fa, ha), (xb, fb, hb), (_, fc, hc) = first_batch(0), first_batch(0), first_batch(1)
print("1) seed 0 lặp lại -> cùng file:", fa == fb, "| cùng ảnh sau aug:", torch.equal(xa, xb), "| cùng head:", torch.equal(ha, hb))
print("   seed 1          -> cùng file:", fa == fc, "| cùng head:", torch.equal(ha, hc))
assert fa == fb and torch.equal(xa, xb) and torch.equal(ha, hb) and fa != fc

# 2) loss CE ban đầu trên một batch val THẬT
val_dl = D.make_loader(val_df, IMAGES_DIR, tf_eval, 64, False)
xv, yv, _ = next(iter(val_dl))
m = M.build_model("resnet50").to(device).eval()
with torch.no_grad():
    l0 = torch.nn.functional.cross_entropy(m(xv.to(device)), yv.to(device)).item()
print(f"2) loss ban đầu (batch val thật) = {l0:.3f}, kỳ vọng ~ {math.log(9):.3f}")

# 3) overfit 18 ảnh train (2 ảnh mỗi lớp, không augmentation) tới loss ~ 0
TR.set_seed(0)
small = train_df.groupby("Label").head(2).reset_index(drop=True)
xs = torch.stack([tf_eval(Image.open(f"{IMAGES_DIR}/{f}").convert("RGB")) for f in small.Filename]).to(device)
ys = torch.tensor(small.Label.values, device=device)
m = M.build_model("resnet50").to(device)
opt = torch.optim.AdamW(M.param_groups(m, 1e-4, 1e-3, 0.0))
hist = []
for step in range(100):
    M.set_train_mode(m)
    loss = torch.nn.functional.cross_entropy(m(xs), ys)
    opt.zero_grad(); loss.backward(); opt.step(); hist.append(loss.item())
m.eval()
with torch.no_grad():
    acc = (m(xs).argmax(1) == ys).float().mean().item()
print(f"3) overfit 18 ảnh: loss {hist[0]:.3f} -> {hist[-1]:.4f} sau 100 bước | acc (eval) {acc:.2f}")
assert hist[-1] < 0.1, "không overfit được batch nhỏ: kiểm tra lại code/model trước khi chạy tiếp"

# 4) ảnh sau augmentation (đã giải chuẩn hoá) + nhãn khớp CSV; CutMix trộn cả ảnh lẫn nhãn
dl = D.make_loader(train_df, IMAGES_DIR, D.build_transforms(True, aug="basic"), 16, True, seed=0)
x, y, f = next(iter(dl))
label_of = train_df.set_index("Filename").Label
assert all(label_of[fi] == int(yi) for fi, yi in zip(f, y)), "nhãn trả về không khớp CSV"
mean = torch.tensor(D.IMAGENET_MEAN).view(3, 1, 1); std = torch.tensor(D.IMAGENET_STD).view(3, 1, 1)
show = lambda t: (t.cpu() * std + mean).clamp(0, 1).permute(1, 2, 0)
xm, (ya, yb, lam) = L.mix_batch(x[:8], y[:8], 1.0, "cutmix")
fig, axes = plt.subplots(3, 8, figsize=(18, 7.5))
for k in range(16):
    a = axes.flat[k]; a.imshow(show(x[k])); a.axis("off")
    a.set_title(f"{D.CLASS_NAMES[int(y[k])]}\n{f[k]}", fontsize=7)
for k in range(8):
    a = axes[2, k]; a.imshow(show(xm[k])); a.axis("off")
    a.set_title(f"CutMix {D.CLASS_NAMES[int(ya[k])]} /\n{D.CLASS_NAMES[int(yb[k])]}, lam={lam:.2f}", fontsize=7)
plt.suptitle("Hàng 1-2: ảnh train sau augmentation 'basic' (đã giải chuẩn hoá). Hàng 3: CutMix")
plt.tight_layout(); plt.savefig("eval_out/check_augmentation.png", dpi=90); plt.show()
print("4) nhãn của 16 ảnh khớp CSV: True")

# 5) train/eval mode: evaluate() đưa model về eval; eval cho kết quả cố định, train (dropout) thì không
m = M.build_model("resnet50", drop_rate=0.5).to(device)
M.set_train_mode(m)
TR.evaluate(m, D.make_loader(val_df.head(16), IMAGES_DIR, tf_eval, 8, False, num_workers=0),
            torch.nn.CrossEntropyLoss(), device)
assert not any(mm.training for mm in m.modules()), "evaluate() phải đặt model.eval()"
x4 = xv[:4].to(device)
with torch.no_grad():
    same_eval = torch.allclose(m(x4), m(x4))
    M.set_train_mode(m)
    same_train = torch.allclose(m(x4), m(x4))
print("5) sau evaluate(): eval mode | eval 2 lần forward giống nhau:", same_eval, "| train mode (dropout) giống nhau:", same_train)
assert same_eval and not same_train
print("\nOK: qua cả 5 kiểm tra pipeline")


In [ ]:
# Chạy thử run(Config(...)) từ đầu đến cuối: 1 epoch, ghi vào thư mục debug (KHÔNG phải kết quả thí nghiệm)
from IPython.display import Image as IPImage
importlib.reload(TR)
dbg = TR.Config(exp_id="B00", desc="smoke", epochs=1, out_dir="eval_out/debug/runs",
                pred_dir="eval_out/debug/predictions", curves_dir="eval_out/debug/curves")
summary = TR.run(dbg)
rd = TR.run_dir(dbg)
print(sorted(p.name for p in rd.iterdir()))
display(pd.read_csv(rd / "history.csv"))
display(pd.read_csv(TR.pred_path(dbg, "val")).head(3))
display(IPImage(filename=str(TR.curve_path(dbg))))
print("parse_overrides:", TR.parse_overrides(["exp_id=B01", "seed=1", "loss=focal", "ema_decay=none", "amp=false", "lr_head=3e-3"]))


In [ ]:
# Bộ test của repo (eval.py + khung code), không cần GPU
!python -m unittest discover -s tests 2>&1 | tail -5


## Ngân sách GPU: đo 1 epoch mỗi backbone
Mỗi backbone chạy **1 epoch train + 1 lượt val** với công thức nền (batch 64, AMP, 224). Không ghi gì ra `runs/`.
Từ đó ước lượng tổng thời gian cho kế hoạch ~20-25 lần huấn luyện và quyết định ablation trên 1 hay 2 backbone (GUIDE mục 7).
Ghi bảng này vào báo cáo; mọi cắt giảm (epoch, số backbone ablation...) cũng phải ghi lại.

In [ ]:
import importlib, train as TR
importlib.reload(TR)
from train import Config, measure_epoch_time

BUDGET_BACKBONES = ["resnet50", "resnext50", "convnext_tiny", "deit_small", "swin_tiny", "efficientnet_b0", "mobilenetv3"]
timing = []
for bb in BUDGET_BACKBONES:
    r = measure_epoch_time(Config(backbone=bb))          # đủ 1 epoch; max_batches=40 để đo nhanh (ngoại suy)
    timing.append(r)
    print(f"{bb:16s} train {r['train_epoch_s']:6.0f}s | val {r['val_s']:5.0f}s | mem {r['peak_mem_GB']:.1f} GB")
timing = pd.DataFrame(timing).set_index("backbone")
timing.to_csv("eval_out/epoch_timing.csv")             # lưu ra Drive
display(timing.round(1))


In [ ]:
# Kế hoạch: sửa số lần chạy cho đúng ý bạn, rồi so với hạn mức GPU (Colab free: không cố định, thường vài giờ/phiên).
EPOCHS = 12
ABLATION_BACKBONES = ["resnet50"]          # 1 backbone; thêm cái thứ 2 nếu tổng giờ còn dư
N_ABLATION_RUNS = 12                       # trục A-G, mỗi lần khác T00 một yếu tố
FINAL_SEEDS = 3                            # F01 x 3 seed + mốc T00 thêm 2 seed (seed0 dùng lại từ Bước 2)

ep = timing["epoch_s"] / 3600              # giờ / epoch
hours = {
    "B: backbones x 1 seed": ep[BUDGET_BACKBONES].sum() * EPOCHS,
    "T: ablation": sum(ep[b] for b in ABLATION_BACKBONES) * EPOCHS * N_ABLATION_RUNS,
    "F: chung kết + mốc": ep[ABLATION_BACKBONES[0]] * EPOCHS * (FINAL_SEEDS + FINAL_SEEDS - 1),
}
plan = pd.Series(hours).round(2); plan["TỔNG (giờ GPU)"] = plan.sum(); plan["+20% chạy hỏng"] = plan.iloc[-1] * 1.2
display(plan.to_frame("giờ"))


## Bước 1 — So sánh backbone (≥ 5)
Cùng công thức nền `T00` (GUIDE.md mục 1.4), cùng seed, mỗi backbone một lần chạy. Mã thí nghiệm `B01`, `B02`, ...

In [ ]:
# Bước 1a: huấn luyện 7 backbone với công thức nền T00, cùng seed 0 (~1,5 giờ trên T4)
# Bị ngắt giữa chừng: chạy lại ô này -> backbone đã xong được bỏ qua, backbone dở dang chạy tiếp từ epoch đã lưu.
!git pull -q
import importlib, dataset as D, model as M, losses as L, train as TR, benchmark as BM
for mod in (D, M, L, TR, BM):
    importlib.reload(mod)

BACKBONE_RUNS = [            # (exp_id, backbone): đủ ResNet, ResNeXt/ConvNeXt, transformer, mạng nhẹ
    ("B01", "resnet50"),
    ("B02", "resnext50"),
    ("B03", "convnext_tiny"),
    ("B04", "deit_small"),
    ("B05", "swin_tiny"),
    ("B06", "efficientnet_b0"),
    ("B07", "mobilenetv3"),
]
for exp_id, bb in BACKBONE_RUNS:
    TR.run(TR.Config(exp_id=exp_id, backbone=bb, desc=bb, seed=0))


In [ ]:
# Bước 1b: bảng Backbones (+ độ trễ sơ bộ batch 1) và biểu đồ đánh đổi. Chạy sau khi Bước 1a xong cả 7.
import json, torch
import matplotlib.pyplot as plt
device = torch.device("cuda")
rows = []
for exp_id, bb in BACKBONE_RUNS:
    cfg = TR.Config(exp_id=exp_id, backbone=bb, desc=bb, seed=0)
    rd = TR.run_dir(cfg)
    s = json.loads((rd / "summary.json").read_text(encoding="utf-8"))
    h = pd.read_csv(rd / "history.csv")
    m = M.build_model(bb, pretrained=False).to(device)
    m.load_state_dict(torch.load(rd / "best.pt", map_location=device, weights_only=True))
    lat = BM.latency_report(m, batch_size=1, img_size=cfg.img_size, dtype="fp32", iters=50)
    rows.append({
        "exp_id": exp_id, "backbone": bb, "tag trọng số": s["weight_tag"],
        "#tham số (M)": round(s["params_M"], 2), "GMAC": round(s["gmacs"], 2),
        "độ phân giải": cfg.img_size, "epoch": cfg.epochs, "seed": cfg.seed,
        "macro-F1 val": round(s["val_macro_f1"], 4), "top-1 val": round(s["val_top1"], 4),
        "F1 Chinee apple val": round(s["val_f1_per_class"][0], 4), "F1 Snake weed val": round(s["val_f1_per_class"][7], 4),
        "best epoch": s["best_epoch"], "val loss min @epoch": int(h.loc[h.val_loss.idxmin(), "epoch"]),
        "train loss cuối": round(h.train_loss.iloc[-1], 4), "val loss cuối": round(h.val_loss.iloc[-1], 4),
        "thời gian train/epoch (s)": round(s["train_time_per_epoch_s"], 1),
        "độ trễ batch-1 p50 (ms)": round(lat["p50"], 2), "độ trễ batch-1 p95 (ms)": round(lat["p95"], 2),
        "ghi chú": f"1 seed; FP32, {lat['gpu']}, không tính tiền xử lý",
    })
    del m; torch.cuda.empty_cache()
backbones = pd.DataFrame(rows)
backbones.to_csv("eval_out/backbones.csv", index=False)       # -> sheet Backbones (lưu trên Drive)
display(backbones.sort_values("macro-F1 val", ascending=False))

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
for a, x, xl in [(ax[0], "độ trễ batch-1 p50 (ms)", "độ trễ batch-1 p50 (ms), FP32"),
                 (ax[1], "GMAC", "GMAC"), (ax[2], "thời gian train/epoch (s)", "thời gian train/epoch (s)")]:
    a.scatter(backbones[x], backbones["macro-F1 val"], s=backbones["#tham số (M)"] * 12, alpha=0.6)
    for _, r in backbones.iterrows():
        a.annotate(r.backbone, (r[x], r["macro-F1 val"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
    a.set(xlabel=xl, ylabel="macro-F1 val", title=f"macro-F1 val theo {xl.split(',')[0]}"); a.grid(alpha=0.3)
fig.suptitle("Bước 1: 7 backbone, công thức nền T00, 1 seed (kích thước điểm ~ số tham số)")
plt.tight_layout(); plt.savefig("eval_out/backbones_tradeoff.png", dpi=120); plt.show()


## Bước 2 — Công thức huấn luyện (≥ 3 trục)
Mỗi lần chạy chỉ khác `T00` **một yếu tố**. Mã `T01`, `T02`, ... Ghi rõ khác ở điểm nào.

In [ ]:
# TODO: các trục A-G của GUIDE.md mục 3 (khởi tạo, augmentation, loss, sampler, LR/optimizer, EMA, ...).
# TODO: thử ít nhất một kết hợp các yếu tố tốt; so Δ với nhiễu (std).

## Bước 3 — Suy luận (≥ 4 phương pháp) và độ trễ
Làm trên **val**. Không huấn luyện lại. Mã `I00` (1 view, mốc), `I01`, ...

In [ ]:
# TODO: TTA lật, multi-crop/scale, dò độ phân giải, gộp xác suất vs logit, ensemble, EMA/soup,
#       temperature scaling (T khớp trên VAL), gộp BN/FP16  (starter/inference.py)
# TODO: độ trễ p50/p95/p99 bằng starter/benchmark.py: warmup, synchronize, >= 50 lần

## Bước 4 — Chung kết: ≥ 3 seed, test một lần mỗi seed
1. Chốt cấu hình trên **val**.
2. Chạy cấu hình cuối **và mốc** (`T00` + `I00`) với cùng số seed, bật `save_test_predictions=True`.
3. Dự đoán lưu ở `predictions/<exp_id>_seed<k>_test.csv`. Rồi chạy `eval.py` (ô dưới).

In [ ]:
# TODO: chạy chung kết và mốc, mỗi seed, ví dụ:
#   for seed in (0, 1, 2):
#       run(Config(exp_id="F01", ..., seed=seed, save_test_predictions=True))
#       run(Config(exp_id="T00", seed=seed, save_test_predictions=True))
# Với suy luận (TTA/ensemble/temperature scaling), tự ghi predictions/F01_seed<k>_test.csv
# bằng eval.save_predictions(...) từ xác suất cuối cùng; thêm F01uncal_* nếu muốn chấm I4(a).

In [ ]:
# Tính chỉ số theo đúng định nghĩa của lớp (đã viết sẵn, không sửa eval.py):
!python {REPO_DIR}/eval.py score --pred "predictions/F01_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag F01 --out eval_out
!python {REPO_DIR}/eval.py score --pred "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag T00 --out eval_out

In [ ]:
# Tự chấm phần I của RUBRIC (đề xuất, giảng viên xác nhận). Thêm --uncal, --final-val, --latency-p95-ms nếu có.
!python {REPO_DIR}/eval.py grade --final "predictions/F01_seed*_test.csv" \
    --baseline "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --out eval_out

## Bước 5 — Sản phẩm
`results.xlsx` (các sheet ở GUIDE.md mục 6.1), `curves/<exp_id>_*.png`, `report.md`, `predictions/`, `code/`.

In [ ]:
# TODO: ghi results.xlsx bằng pandas.ExcelWriter(engine="openpyxl") với các sheet:
#   Backbones, Training, Inference, Final, PerClass, Latency, Summary
# TODO: kiểm tra mỗi exp_id có ảnh trong curves/ và số trong xlsx khớp kết quả của eval.py